In [ ]:
pip install pandas

In [ ]:
import pandas as pd

df = pd.read_csv('../Data/raw/customer_support_tickets.csv')
df.head()

In [ ]:
df.shape

In [ ]:
df.dtypes

In [ ]:
df.isnull().sum()

In [ ]:
# Drop columns we won't use
df.drop(columns=['Customer Name', 'Customer Email', 'Ticket Description', 'Resolution'], inplace=True)

# Parse date column
df['Date of Purchase'] = pd.to_datetime(df['Date of Purchase'])

# Standardise text columns
df['Ticket Status'] = df['Ticket Status'].str.strip().str.title()
df['Ticket Priority'] = df['Ticket Priority'].str.strip().str.title()
df['Ticket Type'] = df['Ticket Type'].str.strip().str.title()
df['Ticket Channel'] = df['Ticket Channel'].str.strip().str.title()

# Check duplicates
print(df.duplicated().sum())

# Save cleaned version
df.to_csv('../Data/processed/tickets_clean.csv', index=False)

print("Cleaning done. Shape:", df.shape)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Overall CSAT
print(f"Average CSAT Score: {df['Customer Satisfaction Rating'].mean():.2f}")
print(f"Total rated tickets: {df['Customer Satisfaction Rating'].count()}")
print("---")
print(df['Customer Satisfaction Rating'].value_counts().sort_index())

In [ ]:
csat_by_type = df.groupby('Ticket Type')['Customer Satisfaction Rating'].mean().sort_values()

print(csat_by_type)

plt.figure(figsize=(8, 5))
csat_by_type.plot(kind='bar', color='#e74c3c')
plt.title('Average CSAT Score by Ticket Type')
plt.xlabel('Ticket Type')
plt.ylabel('Average CSAT (1-5)')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
csat_by_priority = df.groupby('Ticket Priority')['Customer Satisfaction Rating'].mean().sort_values()

print(csat_by_priority)

plt.figure(figsize=(8, 5))
csat_by_priority.plot(kind='bar', color='#e67e22')
plt.title('Average CSAT Score by Ticket Priority')
plt.xlabel('Priority')
plt.ylabel('Average CSAT (1-5)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
csat_by_channel = df.groupby('Ticket Channel')['Customer Satisfaction Rating'].mean().sort_values()

print(csat_by_channel)

plt.figure(figsize=(8, 5))
csat_by_channel.plot(kind='bar', color='#3498db')
plt.title('Average CSAT Score by Channel')
plt.xlabel('Channel')
plt.ylabel('Average CSAT (1-5)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
print(df['Ticket Channel'].value_counts())

In [ ]:
channel_volume = df['Ticket Channel'].value_counts()
print(channel_volume)

plt.figure(figsize=(8, 5))
channel_volume.plot(kind='bar', color='#2ecc71')
plt.title('Ticket Volume by Channel')
plt.xlabel('Channel')
plt.ylabel('Number of Tickets')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
csat_by_product = df.groupby('Product Purchased')['Customer Satisfaction Rating'].mean().sort_values()

print(csat_by_product)

plt.figure(figsize=(10, 6))
csat_by_product.plot(kind='bar', color='#9b59b6')
plt.title('Average CSAT Score by Product')
plt.xlabel('Product')
plt.ylabel('Average CSAT (1-5)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
volume_by_product = df['Ticket Channel'].value_counts()

product_summary = df.groupby('Product Purchased').agg(
    avg_csat=('Customer Satisfaction Rating', 'mean'),
    ticket_count=('Ticket ID', 'count')
).sort_values('avg_csat')

print(product_summary.head(10))

In [ ]:
product_summary = df.groupby('Product Purchased').agg(
    avg_csat=('Customer Satisfaction Rating', 'mean'),
    ticket_count=('Ticket ID', 'count')
).sort_values('avg_csat').reset_index()

product_summary.to_csv('../Data/processed/product_summary.csv', index=False)
print("Saved product summary")

In [ ]:
import sqlite3

# Create a connection to a new SQLite database
conn = sqlite3.connect('../Data/support_tickets.db')

# Load your clean data into a table called 'tickets'
df.to_sql('tickets', conn, if_exists='replace', index=False)

print("Data loaded into SQLite successfully")
print(pd.read_sql("SELECT COUNT(*) as total_rows FROM tickets", conn))

In [ ]:
query1 = """
SELECT 
    "Ticket Channel",
    COUNT(*) as total_tickets,
    ROUND(AVG("Customer Satisfaction Rating"), 2) as avg_csat,
    SUM(CASE WHEN "Ticket Status" = 'Closed' THEN 1 ELSE 0 END) as resolved,
    ROUND(SUM(CASE WHEN "Ticket Status" = 'Closed' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) as resolution_rate_pct
FROM tickets
GROUP BY "Ticket Channel"
ORDER BY avg_csat ASC
"""

result1 = pd.read_sql(query1, conn)
print(result1)

In [ ]:
query2 = """
SELECT 
    "Ticket Type",
    COUNT(*) as total_tickets,
    ROUND(AVG("Customer Satisfaction Rating"), 2) as avg_csat,
    SUM(CASE WHEN "Ticket Status" = 'Closed' THEN 1 ELSE 0 END) as resolved,
    ROUND(SUM(CASE WHEN "Ticket Status" = 'Closed' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) as resolution_rate_pct
FROM tickets
GROUP BY "Ticket Type"
ORDER BY avg_csat ASC
"""

result2 = pd.read_sql(query2, conn)
print(result2)

In [ ]:
query3 = """
WITH product_stats AS (
    SELECT 
        "Product Purchased",
        COUNT(*) as total_tickets,
        ROUND(AVG("Customer Satisfaction Rating"), 2) as avg_csat,
        ROUND(SUM(CASE WHEN "Ticket Status" = 'Closed' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) as resolution_rate_pct
    FROM tickets
    GROUP BY "Product Purchased"
)
SELECT *,
    RANK() OVER (ORDER BY avg_csat ASC) as csat_rank
FROM product_stats
ORDER BY csat_rank
LIMIT 10
"""

result3 = pd.read_sql(query3, conn)
print(result3)

In [ ]:
query4 = """
SELECT 
    "Ticket Priority",
    COUNT(*) as total_tickets,
    ROUND(AVG("Customer Satisfaction Rating"), 2) as avg_csat,
    ROUND(SUM(CASE WHEN "Ticket Status" = 'Closed' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) as resolution_rate_pct
FROM tickets
GROUP BY "Ticket Priority"
ORDER BY avg_csat ASC
"""

result4 = pd.read_sql(query4, conn)
print(result4)

In [ ]:
conn.close()
print("Database connection closed")